# Chapter 7 &mdash; $\varepsilon$-Transitions: Convenient, Not Necessary

**Concept 3 of the Chapter 7 decomposition:** *$\varepsilon$-Transitions, and Whether They Are Essential*

An $\varepsilon$ edge is taken without consuming input &mdash; it simplifies constructions but adds no power.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7/Concept-Epsilon-Transitions/Concept-Epsilon-Transitions.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateNFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


An **$\varepsilon$-transition** moves a token without consuming input. Jove writes it
as `''` in the markdown.

It is a **convenience, not a necessity**: any $\varepsilon$-NFA can be converted to one
without $\varepsilon$ edges by composing each $\varepsilon$ path with the following real
move, and propagating finality backwards along $\varepsilon$ edges.

But the convenience is large. The Thompson constructions of Chapter 8 glue RE
fragments together with $\varepsilon$ edges, and that is what makes them a **two-line**
rule per operator instead of a case analysis.

## 2. Definitions

### An NFA with $\varepsilon$ edges

In [ ]:
E = md2mc('''NFA
I  : '' -> A
I  : '' -> B
A  : 0 -> A
A  : '' -> F
B  : 1 -> B
B  : '' -> F
''')
print("states :", sorted(E["Q"]))
print("Eclosure of the start set :", sorted(Eclosure(E, E["Q0"])))

### The same language without $\varepsilon$, written by hand

In [ ]:
NoE = md2mc('''NFA
IF : 0 -> F0
IF : 1 -> F1
F0 : 0 -> F0
F1 : 1 -> F1
''')

## 3. Tests

An $\varepsilon$ edge is followed without reading anything.

In [ ]:
print("from I, on no input, a token can be at :", sorted(Eclosure(E, {'I'})))
assert 'F' in Eclosure(E, {'I'})
print("so epsilon is accepted :", accepts_nfa(E, ''))

The two machines recognise the same language: $0^* \cup 1^*$.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(9) for p in product('01', repeat=k)]
spec = lambda s: set(s) <= {'0'} or set(s) <= {'1'}
assert all(accepts_nfa(E, s) == spec(s) for s in strs)
assert all(accepts_nfa(NoE, s) == spec(s) for s in strs)
print("both match 0* union 1* on all %d strings up to length 8" % len(strs))
print("langeq after determinizing :", langeq_dfa(min_dfa(nfa2dfa(E)), min_dfa(nfa2dfa(NoE))))
assert langeq_dfa(min_dfa(nfa2dfa(E)), min_dfa(nfa2dfa(NoE)))

So $\varepsilon$ adds **no power** &mdash; both determinize to the same minimal DFA.

In [ ]:
a, b = min_dfa(nfa2dfa(E)), min_dfa(nfa2dfa(NoE))
print("minimal sizes : %d and %d, isomorphic: %s" % (len(a["Q"]), len(b["Q"]), iso_dfa(a, b)))
assert iso_dfa(a, b)

But it does add **convenience**: gluing two machines is one edge, not a rewrite.

In [ ]:
print("with epsilon    : add I' --eps--> I1, I' --eps--> I2.  Two edges, done.")
print("without epsilon : copy every outgoing move of I1 and I2 onto I'.")
print("\nChapter 8's Thompson constructions rely entirely on the first option.")

## 4. Animation

Follow the $\varepsilon$ edges: the token reaches A, B and F before reading a thing.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateNFA import *
AnimateNFA(E, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Remove the $\varepsilon$ edges from `E` yourself. Which state becomes final, and why?
2. Can an $\varepsilon$ **cycle** exist? What does `Eclosure` do with one?
3. Why does finality have to propagate *backwards* along $\varepsilon$ edges?

In [ ]:
# Your work for the exercises above.